In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATH
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "BPQ_C.xpt"
csv_file = DATA_DIR / "BPQ_C_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Keep untouched source for final comparison
source_df = df.copy()


print("=" * 75)
print("NHANES 2003-2004 BPQ_C XPT -> CSV")
print("=" * 75)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))

print("\nVariables:")
print(df.columns.tolist())


# ============================================================
# 3. EXPECTED RELEASED COLUMNS
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "BPQ010",
    "BPQ020",
    "BPQ030",
    "BPQ040A",
    "BPQ040B",
    "BPQ040C",
    "BPQ040D",
    "BPQ040E",
    "BPQ040F",
    "BPQ043A",
    "BPQ043B",
    "BPQ043C",
    "BPQ043D",
    "BPQ050A",
    "BPQ050B",
    "BPQ050C",
    "BPQ050D",
    "BPQ050E",
    "BPQ060",
    "BPQ070",
    "BPQ080",
    "BPQ090A",
    "BPQ090B",
    "BPQ090C",
    "BPQ090D",
    "BPQ100A",
    "BPQ100B",
    "BPQ100C",
    "BPQ100D",
    "BPD110A",
    "BPD110B",
    "BPD110C",
    "BPD120",
    "BPD130",
    "BPD140"
]


if df.columns.tolist() == EXPECTED_COLUMNS:
    print("\nPASS: Column names/order match BPQ_C.")
else:
    raise ValueError(
        "Column structure does not match expected BPQ_C."
    )


# ============================================================
# 4. ZERO / TINY VALUE CHECK
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


true_zero_count = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


total_tiny = int(
    tiny_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    true_zero_count
)

print(
    "Tiny artifact values:",
    total_tiny
)


if total_tiny > 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
    )

    raise ValueError(
        "Unexpected tiny values found in BPQ_C. "
        "Do NOT automatically convert them to zero."
    )


print(
    "PASS: No tiny zero-conversion artifact exists."
)


# ============================================================
# 5. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# BPQ_C should contain questionnaire codes only.
#
# Before converting anything to integer format,
# verify there are no real decimal values.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


for col in numeric_cols:

    values = df[col].dropna()

    if values.empty:
        continue

    fractional = ~np.isclose(
        values,
        np.round(values)
    )

    if fractional.any():

        print(
            f"Unexpected decimal values in {col}:"
        )

        print(
            values[
                fractional
            ].head(20)
        )

        raise ValueError(
            f"{col} contains genuine decimal values. "
            "Column was NOT changed."
        )


print(
    "PASS: All BPQ_C values are whole-number "
    "questionnaire codes/identifiers."
)


# ============================================================
# 6. CLEAN SEQN
# ============================================================

if "SEQN" not in df.columns:

    raise ValueError(
        "SEQN not found."
    )


df["SEQN"] = (
    df["SEQN"]
    .round()
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


# ============================================================
# 7. FORMAT QUESTIONNAIRE CODES WITHOUT .0
# ============================================================

CODE_COLUMNS = [
    col
    for col in df.columns
    if col != "SEQN"
]


for col in CODE_COLUMNS:

    df[col] = (
        df[col]
        .round()
        .astype("Int64")
    )


print(
    "\nPASS: BPQ/BPD questionnaire codes "
    "formatted without unnecessary .0."
)


# ============================================================
# 8. CDC STRUCTURE CHECK
# ============================================================

print("\n--- CDC STRUCTURE CHECK ---")


if len(df) == 6213:
    print("PASS: Row count = 6,213")
else:
    print(
        "FAIL: Row count =",
        len(df)
    )


if len(df.columns) == 36:
    print("PASS: Column count = 36")
else:
    print(
        "FAIL: Column count =",
        len(df.columns)
    )


# ============================================================
# 9. CDC CODE FREQUENCIES
# ============================================================

#
# Frequencies below come from the official BPQ_C codebook.
#
# Zero-frequency codes such as Refused=7 are not listed
# because they do not occur in the actual XPT.
#

EXPECTED = {

    "BPQ010": {
        "counts": {1:4412, 2:868, 3:434, 4:411, 5:75, 9:13},
        "missing": 0
    },

    "BPQ020": {
        "counts": {1:1801, 2:4319, 9:18},
        "missing": 75
    },

    "BPQ030": {
        "counts": {1:1457, 2:339, 9:5},
        "missing": 4412
    },

    "BPQ040A": {
        "counts": {1:1452, 2:347, 9:2},
        "missing": 4412
    },

    "BPQ040B": {
        "counts": {1:867, 2:927, 9:7},
        "missing": 4412
    },

    "BPQ040C": {
        "counts": {1:1157, 2:637, 9:7},
        "missing": 4412
    },

    "BPQ040D": {
        "counts": {1:1095, 2:701, 9:5},
        "missing": 4412
    },

    "BPQ040E": {
        "counts": {1:413, 2:1382, 9:6},
        "missing": 4412
    },

    "BPQ040F": {
        "counts": {1:185, 2:1613, 9:3},
        "missing": 4412
    },

    "BPQ043A": {
        "counts": {1:67},
        "missing": 6146
    },

    "BPQ043B": {
        "counts": {2:4},
        "missing": 6209
    },

    "BPQ043C": {
        "counts": {3:59},
        "missing": 6154
    },

    "BPQ043D": {
        "counts": {4:60},
        "missing": 6153
    },

    "BPQ050A": {
        "counts": {1:1291, 2:160, 9:1},
        "missing": 4761
    },

    "BPQ050B": {
        "counts": {1:678, 2:187, 9:2},
        "missing": 5346
    },

    "BPQ050C": {
        "counts": {1:1008, 2:149},
        "missing": 5056
    },

    "BPQ050D": {
        "counts": {1:649, 2:446},
        "missing": 5118
    },

    "BPQ050E": {
        "counts": {1:324, 2:89},
        "missing": 5800
    },

    "BPQ060": {
        "counts": {1:3486, 2:1378, 9:176},
        "missing": 1173
    },

    "BPQ070": {
        "counts": {1:2357, 2:565, 3:348, 4:192, 9:24},
        "missing": 2727
    },

    "BPQ080": {
        "counts": {1:1510, 2:1935, 9:41},
        "missing": 2727
    },

    "BPQ090A": {
        "counts": {1:1217, 2:282, 9:11},
        "missing": 4703
    },

    "BPQ090B": {
        "counts": {1:826, 2:679, 9:5},
        "missing": 4703
    },

    "BPQ090C": {
        "counts": {1:992, 2:512, 9:6},
        "missing": 4703
    },

    "BPQ090D": {
        "counts": {1:897, 2:609, 9:4},
        "missing": 4703
    },

    "BPQ100A": {
        "counts": {1:978, 2:238, 9:1},
        "missing": 4996
    },

    "BPQ100B": {
        "counts": {1:636, 2:190},
        "missing": 5387
    },

    "BPQ100C": {
        "counts": {1:682, 2:310},
        "missing": 5221
    },

    "BPQ100D": {
        "counts": {1:725, 2:172},
        "missing": 5316
    },

    "BPD110A": {
        "counts": {1:1128, 2:2396, 9:6},
        "missing": 2683
    },

    "BPD110B": {
        "counts": {1:639, 2:2886, 9:5},
        "missing": 2683
    },

    "BPD110C": {
        "counts": {1:598, 2:2928, 9:4},
        "missing": 2683
    },

    "BPD120": {
        "counts": {1:153, 2:140},
        "missing": 5920
    },

    "BPD130": {
        "counts": {1:233, 2:450, 9:1},
        "missing": 5529
    },

    "BPD140": {
        "counts": {1:131, 2:387},
        "missing": 5695
    }
}


# ============================================================
# 10. RUN CDC FREQUENCY CHECK
# ============================================================

print("\n--- CDC FREQUENCY CHECK ---")


frequency_failures = []


for col, spec in EXPECTED.items():

    actual_counts = (
        df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    counts_match = (
        actual_counts
        ==
        spec["counts"]
    )

    missing_match = (
        actual_missing
        ==
        spec["missing"]
    )


    if (
        counts_match
        and
        missing_match
    ):

        print(
            f"PASS: {col}"
        )

    else:

        frequency_failures.append(
            col
        )

        print(
            f"FAIL: {col}"
        )

        print(
            " Expected counts:",
            spec["counts"]
        )

        print(
            " Actual counts:",
            actual_counts
        )

        print(
            " Expected missing:",
            spec["missing"]
        )

        print(
            " Actual missing:",
            actual_missing
        )


if not frequency_failures:

    print(
        "\nPASS: All BPQ_C frequency "
        "distributions match CDC."
    )


# ============================================================
# 11. VERIFY NO NUMERIC VALUES CHANGED
# ============================================================

print("\n--- XPT VALUE FIDELITY BEFORE EXPORT ---")


for col in source_df.columns:

    source_values = (
        source_df[col]
        .to_numpy()
    )

    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        source_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if same:

        print(
            f"PASS: {col:<10} preserved."
        )

    else:

        raise ValueError(
            f"FAIL: {col} changed."
        )


# ============================================================
# 12. EXPORT CSV
# ============================================================

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 13. RE-READ CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False
)


# ============================================================
# 14. XPT -> CSV EXACT VALUE CHECK
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


value_failures = []


for col in source_df.columns:

    xpt_values = (
        source_df[col]
        .to_numpy()
    )

    csv_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        xpt_values,
        csv_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if same:

        print(
            f"PASS: {col:<10} matches XPT."
        )

    else:

        value_failures.append(
            col
        )

        print(
            f"FAIL: {col} differs from XPT."
        )


if not value_failures:

    print(
        "\nPASS: Every BPQ_C value "
        "matches the source XPT."
    )


# ============================================================
# 15. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in source_df.columns:

    xpt_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )

    csv_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            xpt_missing
            !=
            csv_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches == 0:

    print(
        "PASS: Missing values preserved exactly."
    )


# ============================================================
# 16. RAW CSV FORMAT CHECK
# ============================================================

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count == 0:

    print(
        "PASS: No unnecessary .0 formatting."
    )

else:

    print(
        "CHECK: .0 formatting remains."
    )


# ============================================================
# 17. FINAL TINY VALUE CHECK
# ============================================================

remaining_tiny = int(
    csv_df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny artifact values:",
    remaining_tiny
)


if remaining_tiny == 0:

    print(
        "PASS: No tiny zero artifact exists."
    )


print("\n" + "=" * 75)
print("BPQ_C CONVERSION COMPLETE")
print("=" * 75)

NHANES 2003-2004 BPQ_C XPT -> CSV
Rows: 6,213
Columns: 36

Variables:
['SEQN', 'BPQ010', 'BPQ020', 'BPQ030', 'BPQ040A', 'BPQ040B', 'BPQ040C', 'BPQ040D', 'BPQ040E', 'BPQ040F', 'BPQ043A', 'BPQ043B', 'BPQ043C', 'BPQ043D', 'BPQ050A', 'BPQ050B', 'BPQ050C', 'BPQ050D', 'BPQ050E', 'BPQ060', 'BPQ070', 'BPQ080', 'BPQ090A', 'BPQ090B', 'BPQ090C', 'BPQ090D', 'BPQ100A', 'BPQ100B', 'BPQ100C', 'BPQ100D', 'BPD110A', 'BPD110B', 'BPD110C', 'BPD120', 'BPD130', 'BPD140']

PASS: Column names/order match BPQ_C.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact values: 0
PASS: No tiny zero-conversion artifact exists.

--- FRACTIONAL VALUE CHECK ---
PASS: All BPQ_C values are whole-number questionnaire codes/identifiers.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 21005
Maximum: 31125

PASS: BPQ/BPD questionnaire codes formatted without unnecessary .0.

--- CDC STRUCTURE CHECK ---
PASS: Row count = 6,213
PASS: Column count = 36

--- CDC FREQUENCY CHECK ---
PASS: BPQ010
PASS: BPQ020
PASS: